# Aegis.dv – Data Quality & Statistical Validation Audit

**Project:** Aegis.dv – Hospital Operations & Patient Analytics Dashboard  
**Author:** Ananya Biju (Infosys Springboard Internship)  
**Report ID:** DQR-AEGIS-2026-001  

This notebook performs comprehensive data quality auditing, checking completeness, duplicate keys, outlier boundaries, and missingness mechanisms.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

data_path = project_root / 'data' / 'processed' / 'hospital_cleaned.csv'
report_path = project_root / 'reports' / 'data_quality' / 'data_quality_report.json'

df = pd.read_csv(data_path, low_memory=False)
with open(report_path) as f:
    dq_report = json.load(f)

print(f"Loaded processed dataset: {len(df):,} rows x {len(df.columns)} columns")
print(f"Audited Overall Completeness: {dq_report['final_overall_completeness_pct']}%")

## 1. Key Verification Targets & Audit Matrix

In [ ]:
targets = dq_report['targets_achieved']
for rule, passed in targets.items():
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f"{status} - {rule}")

## 2. Statistical Outlier Audit (Tukey IQR Boundaries)

In [ ]:
outlier_df = pd.DataFrame(dq_report['outlier_metrics']).T
print(outlier_df[['q1', 'q3', 'iqr', 'upper_threshold', 'outlier_count', 'outlier_percentage']])

## 3. Inpatient Admission vs Throughput Balance

In [ ]:
admissions = df['encounter_id'].nunique()
discharges = len(df[df['discharge_disposition_id'].notna()])
expired = (df['is_in_hospital_death'] == 1).sum()
routine_home = (df['discharge_disposition_group'] == 'Routine Discharge Home').sum()

print(f"Total Inpatient Admissions: {admissions:,}")
print(f"Total Inpatient Discharges: {discharges:,}")
print(f"Routine Home Discharges:    {routine_home:,} ({routine_home/discharges*100:.2f}%)")
print(f"In-Hospital Mortality:      {expired:,} ({expired/discharges*100:.2f}%)")